# Reinforcement Learning Lab: MDPs, Value Iteration, Q-Learning & Policy Gradients

This interactive notebook demonstrates foundational Reinforcement Learning principles:
1. **Markov Decision Processes (MDP) & Dynamic Programming**: Solving GridWorld via Value Iteration.
2. **Model-Free TD Control**: Tabular Q-Learning with $\epsilon$-greedy exploration.
3. **Policy Gradient**: REINFORCE algorithm optimizing action probability distributions directly.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd() / 'code'))

import numpy as np
from rl_engine import GridWorldMDP, value_iteration, QLearningAgent, REINFORCEPolicyGradient

print('RL modules imported successfully!')

## 1. Environment & Dynamic Programming (Value Iteration)

We set up a $4 \times 4$ GridWorld with a goal state at `(3, 3)` (reward $+10$) and a trap state at `(1, 1)` (reward $-10$). Every intermediate step has a $-0.1$ step penalty to encourage the shortest path.

In [ ]:
env = GridWorldMDP(height=4, width=4, goal=(3, 3), traps=[(1, 1)])
V, policy = value_iteration(env, gamma=0.9, theta=1e-4)

action_symbols = {0: '↑', 1: '↓', 2: '←', 3: '→'}
print('Optimal State Values V*(s):')
print(np.round(V.reshape(4, 4), 2))

print('\nOptimal Policy pi*(s):')
grid_policy = np.array([action_symbols[a] for a in policy]).reshape(4, 4)
print(grid_policy)

## 2. Model-Free TD Control: Q-Learning

Now we assume transition probabilities and rewards are unknown. We train an off-policy **Q-Learning** agent through direct environmental interaction.

In [ ]:
np.random.seed(42)
agent = QLearningAgent(n_states=env.n_states, n_actions=env.n_actions, lr=0.1, gamma=0.9, epsilon=0.3)

n_episodes = 500
episode_rewards = []

for ep in range(n_episodes):
    s = env.reset()
    total_r = 0.0
    done = False
    steps = 0
    while not done and steps < 50:
        a = agent.select_action(s)
        next_s, r, done = env.step(a)
        agent.update(s, a, r, next_s, done)
        s = next_s
        total_r += r
        steps += 1
    episode_rewards.append(total_r)
    agent.epsilon = max(0.01, agent.epsilon * 0.995)

print(f'Average reward (first 50 episodes): {np.mean(episode_rewards[:50]):.2f}')
print(f'Average reward (last 50 episodes):  {np.mean(episode_rewards[-50:]):.2f}')
print('\nMax Q-values per state:')
print(np.round(np.max(agent.Q, axis=1).reshape(4, 4), 2))

## 3. Monte Carlo Policy Gradient (REINFORCE)

In policy gradient methods, we parameterize $\pi_\theta(a \mid s)$ directly using a softmax distribution over action preferences and perform gradient ascent using empirical returns.

In [ ]:
pg_agent = REINFORCEPolicyGradient(n_states=2, n_actions=3, lr=0.1, gamma=0.95)

print('Initial action probabilities for State 0:')
print(np.round(pg_agent.get_action_probs(0), 4))

# Train with trajectories where Action 2 provides high reward
for _ in range(100):
    a = pg_agent.select_action(0)
    reward = 5.0 if a == 2 else -1.0
    trajectory = [(0, a, reward)]
    pg_agent.train_trajectory(trajectory)

print('\nTrained action probabilities for State 0:')
print(np.round(pg_agent.get_action_probs(0), 4))
assert np.argmax(pg_agent.get_action_probs(0)) == 2
print('Successfully converged to optimal action 2!')

## 4. Key Takeaways & Summary

- **Dynamic Programming (Value Iteration)** provides exact, optimal value functions when environmental transition matrices $\mathcal{P}$ and rewards $\mathcal{R}$ are fully specified.
- **Temporal Difference Control (Q-Learning)** allows learning directly from online sample transitions without a world model, balancing exploration and exploitation.
- **Policy Gradient Methods (REINFORCE)** directly optimize policy parameters $\theta$ via gradient ascent on expected return, providing the foundational basis for PPO and modern RLHF.